# CEDAR walkthrough (figure generator)

This notebook is a **worked tour** of the core CEDAR metrics and a reproducible way to regenerate the figures used in the documentation.

- Docs page built from this content: `docs/source/user-guide/cedar-walkthrough.md`
- Figures written to: `docs/source/_static/` (overwrites existing files)

Metrics covered:

- **COP**: coefficient of performance
- **SHR**: sensible heat ratio
- **ECOP**: effective COP = `COP × SHR`
- **CDD**: cooling degree days exceedance
- **eCDD**: effective CDD = `CDD / (COP × SHR)` = `CDD / ECOP`

```{note}
All temperatures are **Kelvin** and relative humidity is **0–1**.
```


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from cedar.metrics.cdd import CoolingDegreeDays
from cedar.metrics.cop import SingleFluidCOP
from cedar.metrics.effective_cdd import EffectiveCDD
from cedar.metrics.effective_cop import EffectiveCOP
from cedar.metrics.shr import SensibleHeatRatioModel


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / "pyproject.toml").exists():
            return path
    return start


ROOT = find_repo_root()
STATIC_DIR = ROOT / "docs" / "source" / "_static"
STATIC_DIR.mkdir(parents=True, exist_ok=True)


def savefig(fig: plt.Figure, filename: str, *, dpi: int = 200) -> Path:
    path = STATIC_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path


np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 200,
        "axes.grid": True,
        "grid.linestyle": "--",
        "grid.alpha": 0.35,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
    }
)

STATIC_DIR


## 1) Sensible Heat Ratio (SHR)

`SensibleHeatRatioModel` computes SHR from air temperature plus either:

- `RH_array=...` (relative humidity, 0–1), **or**
- `dewpoint_array=...` (dew point in K; the model converts dew point → RH internally)

Higher humidity generally reduces SHR (more latent load).


In [ ]:
shr_model = SensibleHeatRatioModel(
    p_atm=101_325.0,
    T_evap=273.15,
    approach_temp=1.0,
    C_p=1020.05,
    H_fg=2.501e6,
    RH_out=1.0,
)
shr_model


In [ ]:
# SHR surface: temperature (x) vs relative humidity (y)
temp_axis = np.linspace(283.15, 333.15, 120)  # 10°C → 60°C
rh_axis = np.linspace(0.0, 1.0, 100)
temp_grid, rh_grid = np.meshgrid(temp_axis, rh_axis)

shr_grid = shr_model.shr(temp_grid, RH_array=rh_grid)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
mesh = ax.pcolormesh(
    temp_axis,
    rh_axis,
    shr_grid,
    shading="auto",
    cmap="viridis",
    vmin=0,
    vmax=1,
)

contours = ax.contour(
    temp_axis,
    rh_axis,
    shr_grid,
    levels=np.arange(0.2, 1.01, 0.1),
    colors="k",
    linewidths=0.7,
    alpha=0.55,
)
ax.clabel(contours, fmt="%.1f", fontsize=8)
fig.colorbar(mesh, ax=ax, pad=0.02, label="SHR")

ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Relative Humidity (-)")
ax.set_title("SHR vs Temperature and Relative Humidity")

fig.tight_layout()
savefig(fig, "shr.png")
plt.show()


## 2) Coefficient of Performance (COP)

`SingleFluidCOP` is a fast, vectorized COP calculator for one refrigerant and one operating-point definition. It precomputes a dense lookup table once, then interpolates for arbitrary-shaped ambient temperature arrays.

The parameters below define a single operating point (evaporator temperature, condenser approach, efficiency, and a minimum lift constraint).


In [ ]:
hp = SingleFluidCOP(
    "R134a",
    T_evap_K=273.15,
    delta_T_cond=10,
    eta_is=0.8,
    delta_T_min=10.0,
)
hp


In [ ]:
# COP reference curve (saved for the docs)
_ = hp.plot(show=True, save_path=STATIC_DIR / "cop.png", dpi=200, figsize=(7.2, 4.6))


## 3) Use case: compare system archetypes

Below is a simple pattern for comparing multiple refrigerants / operating assumptions over the same ambient profile. Treat these as **illustrative** “archetypes” (not calibrated product models).


In [ ]:
ambient_profile = np.linspace(280.0, 305.0, 80)  # K

configs = [
    # Comfort cooling
    dict(
        label="Residential AC",
        refrigerant="R410A",
        T_evap_K=278.15,  # ~5°C
        delta_T_cond=10,
        eta_is=0.75,
        delta_T_min=10.0,
    ),
    dict(
        label="High-efficiency AC",
        refrigerant="R32",
        T_evap_K=279.15,
        delta_T_cond=8,
        eta_is=0.80,
        delta_T_min=8.0,
    ),
    # Commercial refrigeration
    dict(
        label="Commercial Chiller",
        refrigerant="R134a",
        T_evap_K=273.15,  # 0°C
        delta_T_cond=12,
        eta_is=0.78,
        delta_T_min=12.0,
    ),
    # Cold storage
    dict(
        label="Walk-in Freezer",
        refrigerant="R404A",
        T_evap_K=248.15,  # −25°C
        delta_T_cond=15,
        eta_is=0.70,
        delta_T_min=15.0,
    ),
    # CO₂ systems (illustrative; transcritical operation is more complex in reality)
    dict(
        label="CO₂ (illustrative)",
        refrigerant="R744",
        T_evap_K=268.15,
        delta_T_cond=20,
        eta_is=0.65,
        delta_T_min=20.0,
    ),
    # Heat pump (illustrative)
    dict(
        label="Air-source Heat Pump",
        refrigerant="R134a",
        T_evap_K=263.15,  # −10°C
        delta_T_cond=8,
        eta_is=0.80,
        delta_T_min=8.0,
    ),
]

fig, ax = plt.subplots(figsize=(7.8, 4.8))
summary = []

for cfg in configs:
    model = SingleFluidCOP(
        cfg["refrigerant"],
        T_evap_K=cfg["T_evap_K"],
        delta_T_cond=cfg["delta_T_cond"],
        eta_is=cfg["eta_is"],
        delta_T_min=cfg["delta_T_min"],
    )
    cop_vals = model.cop(ambient_profile)
    ax.plot(ambient_profile, cop_vals, lw=2.0, label=f"{cfg['label']} ({cfg['refrigerant']})")
    summary.append((cfg["label"], cfg["refrigerant"], float(np.nanmean(cop_vals))))

ax.set_xlabel("Ambient temperature (K)")
ax.set_ylabel("COP")
ax.set_title("COP profiles for common system archetypes")
ax.set_ylim(bottom=0)
ax.legend(fontsize=8, ncol=2, frameon=False)

fig.tight_layout()
savefig(fig, "multi-refrigerant.png")
plt.show()

for label, refrigerant, mean_cop in summary:
    print(f"{label:22} {refrigerant:5}  mean COP ≈ {mean_cop:.2f}")


## 4) Effective COP (ECOP = COP × SHR)

ECOP folds humidity into an “effective” efficiency proxy. Here we reuse the `hp` and `shr_model` instances so the combined metric reflects one consistent set of assumptions.


In [ ]:
ecop_model = EffectiveCOP(cop_model=hp, shr_model=shr_model)
ecop_model


In [ ]:
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import FuncFormatter

# Grid for the ECOP surface
temp_axis = np.linspace(283.15, 323.15, 140)  # K
rh_axis = np.linspace(0.0, 1.0, 100)
temp_grid, rh_grid = np.meshgrid(temp_axis, rh_axis)

ecop_grid = ecop_model.ecop(temp_grid, RH_array=rh_grid)

finite = np.isfinite(ecop_grid) & (ecop_grid > 0)
vmin = float(np.nanmin(ecop_grid[finite])) if np.any(finite) else 0.0
vmax = float(np.nanmax(ecop_grid[finite])) if np.any(finite) else 1.0

ecop_plot = ecop_grid.copy()
ecop_plot[~finite] = vmin  # avoids NaN/inf rendering gaps

# Discrete bins every 0.5 (nice for reading off values)
start = np.floor(vmin * 2) / 2
stop = np.ceil(vmax * 2) / 2
levels = np.arange(start, stop + 0.5, 0.5)

cmap = plt.get_cmap("viridis").copy()
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

fig, ax = plt.subplots(figsize=(7.6, 4.8))
cf = ax.contourf(
    temp_grid,
    rh_grid,
    ecop_plot,
    levels=levels,
    cmap=cmap,
    norm=norm,
    antialiased=False,
)

ax.contour(temp_grid, rh_grid, ecop_plot, levels=levels, colors="k", linewidths=0.45, alpha=0.35)

label_levels = np.arange(np.ceil(start / 2) * 2, stop + 0.001, 2.0)
cs2 = ax.contour(temp_grid, rh_grid, ecop_plot, levels=label_levels, colors="k", linewidths=0.9, alpha=0.75)
ax.clabel(cs2, fmt="%g", fontsize=8)


def nice_half_formatter(x, _pos):
    if abs(x - round(x)) < 1e-9:
        return f"{int(round(x))}"
    return f"{x:.1f}".rstrip("0").rstrip(".")


cbar_ticks = np.arange(start, stop + 0.001, 1.0)
cbar = fig.colorbar(cf, ax=ax, pad=0.02, ticks=cbar_ticks)
cbar.ax.yaxis.set_major_formatter(FuncFormatter(nice_half_formatter))
cbar.set_label("Effective Coefficient of Performance (ECOP)")

ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Relative Humidity (–)")
ax.set_title("ECOP as a Function of Temperature and Relative Humidity")
ax.set_xlim(temp_axis.min(), temp_axis.max())
ax.set_ylim(0, 1)

fig.tight_layout()
savefig(fig, "ecop.png")
plt.show()


## 5) Cooling Degree Days (CDD)

CDD is a simple exceedance proxy: how far above a base temperature the ambient air is. This example uses a base of **18°C**.


In [ ]:
cdd_model = CoolingDegreeDays(base_temperature_K=18.0 + 273.15)

ambient_profile = np.linspace(280.0, 305.0, 100)  # K
cdd_vals = cdd_model.cdd(ambient_profile)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(ambient_profile, cdd_vals, linewidth=2.8)

ax.set_title("CDD reference chart – base 18°C", pad=10)
ax.set_xlabel("Ambient temperature (K)")
ax.set_ylabel("CDD exceedance")

ax.set_xlim(ambient_profile.min(), ambient_profile.max())
ax.set_ylim(0, max(1e-6, float(np.max(cdd_vals)) * 1.05))

fig.tight_layout()
savefig(fig, "cdd.png")
plt.show()


## 6) Effective Cooling Degree Days (eCDD = CDD / ECOP)

eCDD incorporates both efficiency and humidity through `ECOP = COP × SHR`:

`eCDD = CDD / (COP × SHR) = CDD / ECOP`

Interpretation: higher eCDD implies more “effective” cooling demand given the assumed system performance.


In [ ]:
# Reuse the same COP/SHR assumptions used above
ecdd_model = EffectiveCDD(cdd_model=cdd_model, cop_model=hp, shr_model=shr_model)

ecdd_vals = ecdd_model.ecdd(temp_grid, RH_array=rh_grid)

finite = np.isfinite(ecdd_vals) & (ecdd_vals >= 0)
vmin = float(np.nanmin(ecdd_vals[finite])) if np.any(finite) else 0.0
vmax = float(np.nanmax(ecdd_vals[finite])) if np.any(finite) else 1.0

ecdd_plot = ecdd_vals.copy()
ecdd_plot[~finite] = vmin

bin_step = 1.0
start = np.floor(vmin / bin_step) * bin_step
stop = np.ceil(vmax / bin_step) * bin_step
levels = np.arange(start, stop + bin_step, bin_step)

cmap = plt.get_cmap("viridis").copy()
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

fig, ax = plt.subplots(figsize=(7.8, 4.8))
cf = ax.contourf(
    temp_grid,
    rh_grid,
    ecdd_plot,
    levels=levels,
    cmap=cmap,
    norm=norm,
    antialiased=False,
)

ax.contour(temp_grid, rh_grid, ecdd_plot, levels=levels, colors="k", linewidths=0.35, alpha=0.22)

major_step = 5.0
major_levels = np.arange(np.ceil(start / major_step) * major_step, stop + 0.001, major_step)
cs_major = ax.contour(temp_grid, rh_grid, ecdd_plot, levels=major_levels, colors="k", linewidths=0.9, alpha=0.70)
ax.clabel(cs_major, fmt="%g", fontsize=8, inline=True, inline_spacing=3)


def nice_formatter(x, _pos):
    if abs(x - round(x)) < 1e-9:
        return f"{int(round(x))}"
    return f"{x:.2f}".rstrip("0").rstrip(".")


cbar_ticks = np.arange(np.ceil(start / major_step) * major_step, stop + 0.001, major_step)
cbar = fig.colorbar(cf, ax=ax, pad=0.02, ticks=cbar_ticks)
cbar.ax.yaxis.set_major_formatter(FuncFormatter(nice_formatter))
cbar.set_label("Effective Cooling Degree Days (eCDD)")

ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Relative Humidity (–)")
ax.set_title("eCDD as a Function of Temperature and Relative Humidity")
ax.set_xlim(temp_axis.min(), temp_axis.max())
ax.set_ylim(0, 1)

fig.tight_layout()
savefig(fig, "ecdd.png")
plt.show()


## Next steps

- For runnable scripts (and CSV outputs), see `examples/`.
- For the polished narrative docs version, see `docs/source/user-guide/cedar-walkthrough.md`.
